In [1]:
import sys
from pathlib import Path

print(sys.executable)
print(Path.cwd())

c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\.venv\Scripts\python.exe
c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\notebooks


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(7043, 21)

In [4]:
target_col = "Churn"

y = df[target_col].map({
    "No": 0,
    "Yes": 1,
})

y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [5]:
y.value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [6]:
assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}

Ячейка 5 — create feature matrix

In [7]:
id_cols = ["customerID"]

X = df.drop(columns=id_cols + [target_col])

X.shape

(7043, 19)

In [8]:
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

In [9]:
X[["tenure", "MonthlyCharges", "TotalCharges"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7043 non-null   int64  
 1   MonthlyCharges  7043 non-null   float64
 2   TotalCharges    7032 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 165.2 KB


In [10]:
X["TotalCharges"].isna().sum()

np.int64(11)

In [11]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

all_features = numeric_features + categorical_features

len(numeric_features), len(categorical_features), len(all_features), X.shape[1]

(4, 15, 19, 19)

In [12]:
set(all_features) == set(X.columns)

True

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [14]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5634, 19), (1409, 19), (5634,), (1409,))

In [15]:
split_distribution = pd.DataFrame({
    "full": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

split_distribution

,full,train,test
Churn,,,
0,0.73463,0.734647,0.734564
1,0.26537,0.265353,0.265436


In [16]:
split_counts = pd.DataFrame({
    "full": y.value_counts().sort_index(),
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
})

split_counts

,full,train,test
Churn,,,
0,5174,4139,1035
1,1869,1495,374


## Stage 3 setup notes

- Task type: binary classification.
- Target: `Churn`.
- Positive class: `Yes`, encoded as `1`.
- Negative class: `No`, encoded as `0`.
- `customerID` is excluded from `X`.
- `Churn` is excluded from `X`.
- `TotalCharges` is converted to numeric with `errors="coerce"`.
- `TotalCharges` missing values are not imputed before split.
- Train/test split reuses the same logic as Stage 2:
  - `test_size=0.2`
  - `random_state=42`
  - `stratify=y`
- `X_test` and `y_test` are created but must stay untouched during Stage 3.
- Model selection will use only `X_train` and `y_train`.
- Baseline models will be evaluated with 5-fold `StratifiedKFold` cross-validation.

Ячейка 11 — imports for preprocessing, models and metrics

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate

Ячейка 12 — preprocessing pipelines

In [20]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

Да, scaler здесь включаем. Для Logistic Regression он полезен. Для дерева не обязателен, но на baseline-этапе это допустимо: дерево просто не выиграет от scaler, но и критично не сломается.

Ячейка 13 — define baseline models

In [21]:
models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Logistic Regression balanced": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "Decision Tree max_depth=3": DecisionTreeClassifier(
        max_depth=3,
        random_state=42,
    ),
}

models

{'Dummy most_frequent': DummyClassifier(strategy='most_frequent'),
 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Logistic Regression balanced': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'Decision Tree max_depth=3': DecisionTreeClassifier(max_depth=3, random_state=42)}

Ячейка 14 — CV setup and metrics

In [22]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

Ячейка 15 — cross-validation loop

In [23]:
cv_results = []

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    
    result = {"model": model_name}
    
    for metric_name in scoring.keys():
        result[f"{metric_name}_mean"] = scores[f"test_{metric_name}"].mean()
        result[f"{metric_name}_std"] = scores[f"test_{metric_name}"].std()
    
    cv_results.append(result)

results_df = pd.DataFrame(cv_results)

results_df

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Dummy most_frequent,0.734647,0.000094,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000
1,Logistic Regression,0.802096,0.011584,0.652899,0.025457,0.543144,0.041103,0.592316,0.029612,0.846137,0.012545
2,Logistic Regression balanced,0.748138,0.015307,0.516482,0.018901,0.801338,0.037933,0.627927,0.023225,0.845950,0.012379
3,Decision Tree max_depth=3,0.789494,0.004916,0.699093,0.020446,0.363880,0.025593,0.477956,0.021607,0.816043,0.006982


Ячейка 16 — cleaner comparison table

In [25]:
comparison_cols = [
    "model",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
]

baseline_comparison = (
    results_df[comparison_cols]
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
)

baseline_comparison

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
0,Logistic Regression balanced,0.748138,0.516482,0.801338,0.627927,0.845950
1,Logistic Regression,0.802096,0.652899,0.543144,0.592316,0.846137
2,Decision Tree max_depth=3,0.789494,0.699093,0.363880,0.477956,0.816043
3,Dummy most_frequent,0.734647,0.000000,0.000000,0.000000,0.500000


Ячейка 17 — rounded table

In [26]:
baseline_comparison_rounded = baseline_comparison.copy()

metric_cols = [
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
]

baseline_comparison_rounded[metric_cols] = baseline_comparison_rounded[metric_cols].round(4)

baseline_comparison_rounded

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459
1,Logistic Regression,0.8021,0.6529,0.5431,0.5923,0.8461
2,Decision Tree max_depth=3,0.7895,0.6991,0.3639,0.4780,0.8160
3,Dummy most_frequent,0.7346,0.0000,0.0000,0.0000,0.5000


## Stage 3 baseline conclusions

### Evaluation setup

- All baseline models were evaluated only on `X_train` / `y_train`.
- `X_test` / `y_test` were not used.
- Evaluation method: 5-fold `StratifiedKFold` cross-validation.
- Preprocessing was placed inside sklearn `Pipeline`.
- Numeric preprocessing:
  - `SimpleImputer(strategy="median")`
  - `StandardScaler()`
- Categorical preprocessing:
  - `SimpleImputer(strategy="most_frequent")`
  - `OneHotEncoder(handle_unknown="ignore")`
- No hyperparameter tuning was performed.
- No threshold tuning was performed.

### Baseline results

- `DummyClassifier(strategy="most_frequent")` predicts only the majority class.
  - Accuracy is about 0.735.
  - Recall, precision and F1 for churn class are 0.
  - ROC-AUC is 0.5.
- `Logistic Regression balanced` has the best F1-score:
  - F1 ≈ 0.628.
- `Logistic Regression balanced` also has the best recall:
  - recall ≈ 0.801.
- Standard `Logistic Regression` has the best ROC-AUC:
  - ROC-AUC ≈ 0.846.
- Standard `Logistic Regression` also has higher precision than the balanced version:
  - precision ≈ 0.653 vs. ≈ 0.517.
- `Decision Tree max_depth=3` has relatively high precision but weak recall:
  - precision ≈ 0.699;
  - recall ≈ 0.364.

### Interpretation

- `class_weight="balanced"` makes Logistic Regression pay more attention to the minority class `Churn = Yes`.
- This improves recall because the model catches more churn customers.
- The cost is lower precision because the model also produces more false positives.
- For a churn task, higher recall may be valuable if missing a churn customer is more expensive than offering retention actions to some customers who would not churn.
- Standard Logistic Regression has almost the same ROC-AUC as the balanced version, which means their ranking quality is very similar.
- The difference between them is mainly in the default classification threshold behavior.

### Current baseline verdict

- Best baseline by F1: `Logistic Regression balanced`.
- Best baseline by recall: `Logistic Regression balanced`.
- Best baseline by ROC-AUC: standard `Logistic Regression`.
- A reasonable candidate for the next stage is `Logistic Regression balanced`, especially if the business priority is detecting churn cases.
- However, final model choice should not be based on Stage 3 alone.
- Later stages may compare additional models, tune hyperparameters, and tune threshold using validation/CV only.